# Import libs

In [1]:
from tokenizers import Tokenizer, pre_tokenizers, trainers, models
from datasets import load_dataset

ds = load_dataset("thainq107/iwslt2015-en-vi")

# Tokenize / Preprocessing

In [2]:
# word - based
tokenizer_en = Tokenizer(models.WordLevel(unk_token="<unk>"))
tokenizer_vi = Tokenizer(models.WordLevel(unk_token="<unk>"))
tokenizer_en.pre_tokenizer = pre_tokenizers.Whitespace()
tokenizer_vi.pre_tokenizer = pre_tokenizers.Whitespace()
trainer = trainers.WordLevelTrainer(
    vocab_size=15000,
    min_frequency=2,
    special_tokens=["<pad>", "<unk>", "<bos>", "<eos>"],
)
# train tokenizer
tokenizer_en.train_from_iterator(ds["train"]["en"], trainer)
tokenizer_vi.train_from_iterator(ds["train"]["vi"], trainer)
# tokenizer
tokenizer_en.save("tokenizer_en.json")
tokenizer_vi.save("tokenizer_vi.json")

# Build vocabulary

In [4]:
from transformers import PreTrainedTokenizerFast

MAX_LEN = 75

# Load tokenizer
tokenizer_en = PreTrainedTokenizerFast(
    tokenizer_file="tokenizer_en.json",
    unk_token="<unk>",
    pad_token="<pad>",
    bos_token="<bos>",
    eos_token="<eos>",
)

tokenizer_vi = PreTrainedTokenizerFast(
    tokenizer_file="tokenizer_vi.json",
    unk_token="<unk>",
    pad_token="<pad>",
    bos_token="<bos>",
    eos_token="<eos>",
)

def preprocess_function(examples):
    src_texts = examples["en"]
    tgt_texts = ["<bos> " + sent + "<eos>" for sent in examples["vi"]]
    src_encodings = tokenizer_en(
        src_texts, padding="max_length", truncation=True, max_length=MAX_LEN
    )
    tgt_encodings = tokenizer_vi(
        tgt_texts, padding="max_length", truncation=True, max_length=MAX_LEN
    )
    return {
        "input_ids": src_encodings["input_ids"],
        "labels": tgt_encodings["input_ids"],
    }


preprocessed_ds = ds.map(preprocess_function, batched=True)

c:\Users\Admin\miniconda3\envs\tf-gpu\lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Map:   0%|          | 0/133317 [00:00<?, ? examples/s]

Map:   0%|          | 0/1268 [00:00<?, ? examples/s]

Map:   0%|          | 0/1268 [00:00<?, ? examples/s]

In [18]:
preprocessed_ds['train']['labels'][0]

[2,
 1960,
 66,
 1157,
 131,
 8,
 376,
 113,
 38,
 417,
 735,
 3,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0]

# Modeling

In [19]:
import torch
import torch.nn as nn
from transformers import PreTrainedModel, PretrainedConfig

In [48]:
class Seq2SeqRNNConfig(PretrainedConfig):
    def __init__(
        self,
        vocab_size_src=10000,
        vocab_size_tgt=10000,
        embedding_dim=128,
        hidden_size=128,
        drop_out=0.1,
        **kwargs
    ):
        super().__init__(**kwargs)
        self.vocab_size_src = vocab_size_src
        self.vocab_size_tgt = vocab_size_tgt
        self.embedding_dim = embedding_dim
        self.hidden_size = hidden_size
        self.drop_out = drop_out


class EncoderRNN(nn.Module):
    def __init__(self, input_size, embedding_dim, hidden_size, drop_out=0.1):
        super(EncoderRNN, self).__init__()
        self.embedding = nn.Embedding(
            input_size, embedding_dim
        )  # input_size = vn_vocab_size
        self.hidden_size = hidden_size
        self.gru = nn.GRU(embedding_dim, hidden_size, batch_first=True)
        self.dropout = nn.Dropout(drop_out)

    def forward(self, x):
        embedded = self.embedding(x)
        embedded = self.dropout(embedded)
        output, hidden = self.gru(embedded)
        return output, hidden  # B x S x H, B x H


class DecoderRNN(nn.Module):
    def __init__(self, hidden_size, embedding_dim, output_size):
        super(DecoderRNN, self).__init__()
        self.embedding = nn.Embedding(
            output_size, embedding_dim
        )  # output_size = en_vocab_size
        self.gru = nn.GRU(embedding_dim, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x, hidden):
        embedded = self.embedding(x)
        output, hidden = self.gru(embedded, hidden)  # with hidden is h0
        output = self.fc(output)
        return output, hidden


class Seq2SeqRNNModel(PreTrainedModel):
    config_class = Seq2SeqRNNConfig

    def __init__(self, config, tokenizer_en):
        super().__init__(config)
        self.encoder = EncoderRNN(
            config.vocab_size_src,
            config.embedding_dim,
            config.hidden_size,
            config.drop_out,
        )
        self.decoder = DecoderRNN(
            config.hidden_size, config.embedding_dim, config.vocab_size_tgt
        )
        self.BOS_IDX = tokenizer_en.bos_token_id
        self.loss_fn = nn.CrossEntropyLoss(ignore_index=tokenizer_en.pad_token_id)

    def forward(self, input_ids, labels):
        batch_size, seq_len = labels.shape  # get batch_size and seq_len
        decoder_input = torch.full((batch_size, 1), self.BOS_IDX, dtype=torch.long).to(input_ids.device)  # generate "<bos>" token for a batch
        encoder_output, encoder_hidden = self.encoder(input_ids)
        decoder_outputs = []

        for i in range(seq_len):
            decoder_output, decoder_hidden = self.decoder(decoder_input, encoder_hidden)
            decoder_outputs.append(decoder_output)
            decoder_input = labels[:, i].unsqueeze(1)  # shift left (teach forcing)

        logits = torch.cat(decoder_outputs, dim=1)  # B x S x Vocab
        loss = self.loss_fn(logits.view(-1, logits.shape[-1]), labels.view(-1))
        return {"loss": loss, "logits": logits}


config = Seq2SeqRNNConfig(
    vocab_size_src=len(tokenizer_en), vocab_size_tgt=len(tokenizer_vi)
)
model = Seq2SeqRNNModel(config, tokenizer_en)

# Testing

In [49]:
input_ids = torch.tensor([preprocessed_ds["train"][0]["input_ids"]])
labels = torch.tensor([preprocessed_ds["train"][0]["input_ids"]])
pred = model(input_ids, labels)

In [50]:
pred

{'loss': tensor(9.6502, grad_fn=<NllLossBackward0>),
 'logits': tensor([[[ 0.3463,  0.0399, -0.0219,  ..., -0.0939, -0.0024,  0.0515],
          [ 0.3223, -0.0145,  0.0988,  ...,  0.3382,  0.4115,  0.0205],
          [-0.0048, -0.0858, -0.1741,  ...,  0.2418,  0.1629,  0.0834],
          ...,
          [ 0.1811, -0.1769, -0.2192,  ...,  0.1480,  0.2351,  0.2015],
          [ 0.1811, -0.1769, -0.2192,  ...,  0.1480,  0.2351,  0.2015],
          [ 0.1811, -0.1769, -0.2192,  ...,  0.1480,  0.2351,  0.2015]]],
        grad_fn=<CatBackward0>)}

# Trainer

In [53]:
# Disable wandb
import os

os.environ["WANDB_DISABLED"] = "true"
from transformers import Trainer, TrainingArguments

# Training
training_args = TrainingArguments(
    output_dir="./en-vi-machine-translation",
    logging_dir="logs",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    per_device_train_batch_size=512,
    per_device_eval_batch_size=512,
    num_train_epochs=25,
    learning_rate=2e-5,
    save_total_limit=1,
    report_to="wandb",
)